In [2]:
import os
from dotenv import load_dotenv
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma

import numpy as np
from typing import List

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel


load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [ ]:
loader = DirectoryLoader(
    path="../data/txt",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"},
    show_progress=True
)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

chunks = splitter.split_documents(documents=documents)

store_location = "../vector_store/chroma_db"
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')


vector_store = Chroma.from_documents(
    documents=chunks,
    collection_name="basic_rag_collection2",
    embedding=embeddings_model,
    persist_directory=store_location
)

print(f"Number of vectors created: {vector_store._collection.count()}")
retriever = vector_store.as_retriever()

Number of vectors created: 34


In [3]:
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. If you dont know the answer based on the context, say you dont know. 
                                                 
                                                 Provide specific details from the context to support your answer.
                                                 
                                                 Context: {context}
                                                 Question: {question}
                                                 Answer:""")

In [15]:
llm = ChatOpenAI(model_name='gpt-4o-mini', max_tokens=500)

In [13]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = (
    {"context":retriever | format_docs,
     "question": RunnablePassthrough()}
     | custom_prompt
     | llm
     | StrOutputParser() 
)

response = rag_chain.invoke("What are the documents required to purchase farmland?")
print(response)

In [ ]:
retriever.invoke(input="What are the documents required to purchase farmland?")